# Inheritance, interfaces, and useful warnings

**Lesson 3 of 3 | Intermediate | About 45 minutes**

## Learning scenario

A robot-cell document is structurally valid and all of its class
paths resolve. Yet two instances do not carry the complete shape
expected from their SystemUnitClasses. We will determine whether
this is intentional 150% modeling or an omission, then repair it.


## 1. Load the warning-focused example

The catalog file name tells us its size, subject, polarity, and the
warning types it is designed to demonstrate.


In [ ]:
from automationml import load_json
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "examples").exists():
    repo_root = repo_root.parent

example_path = repo_root / (
    "examples/validation-suite/json/"
    "medium_robot-cell_warning_missing-inherited-attribute-and-class-interface.json"
)
document = load_json(example_path)


In [ ]:
cell = document.instance_hierarchies[0].internal_elements[0]
robot, vision_gate = cell.internal_elements

print(cell.name)
print("  +--", robot.name, "->", robot.ref_base_system_unit_path)
print("  +--", vision_gate.name, "->", vision_gate.ref_base_system_unit_path)


## 2. Follow the robot's class path

`RobotArm` references `RobotModule`. That class inherits from
`Cell`, so its effective shape combines knowledge from both classes.


In [ ]:
library = document.system_unit_class_libs[0]
classes = {item.name: item for item in library.system_unit_classes}

cell_class = classes["Cell"]
robot_class = classes["RobotModule"]
quality_class = classes["QualityGateModule"]

print("RobotModule inherits from", robot_class.ref_base_class_path)


## 3. Compare declared and materialized members

`cycleTime` comes from the base `Cell` class. `payload` and the four
robot interfaces are declared directly on `RobotModule`.


In [ ]:
print("Cell attributes:", [item.name for item in cell_class.attributes])
print("RobotModule attributes:", [item.name for item in robot_class.attributes])
print("RobotArm attributes:", [item.name for item in robot.attributes])


In [ ]:
print(
    "RobotModule interfaces:",
    [item.name for item in robot_class.external_interfaces],
)
print(
    "RobotArm interfaces:",
    [item.name for item in robot.external_interfaces],
)


<details>
<summary><strong>Predict the warning profile before running validation</strong></summary>

Both child instances omit inherited `cycleTime`: two
`missing-inherited-attribute` warnings. RobotArm omits `Power` and
`MaterialOut`; VisionGate omits `MaterialOut`: three
`missing-class-interface` warnings.
</details>

## 4. Validate the prediction


In [ ]:
from collections import Counter

issues = document.caex_validation_issues(strict_xsd=True)
counts = Counter(issue.code for issue in issues)

print(counts)
for issue in issues:
    print(f"{issue.severity.upper():7} {issue.code}: {issue.target}")


In [ ]:
expected = Counter({
    "missing-inherited-attribute": 2,
    "missing-class-interface": 3,
})

assert counts == expected
assert all(issue.severity == "warning" for issue in issues)
print("Prediction confirmed: five warnings and no errors.")


## 5. Why these are warnings

A class can deliberately describe a 150% set of possible members;
an occurrence may use only part of it. The SDK therefore reports
missing materialization as reviewable warnings. Unresolved class
paths, by contrast, are errors because their meaning cannot be found.

We will assume this project expects full materialization.


## 6. Repair the inherited attributes

We work on a deep copy so the warning example remains available for
comparison.


In [ ]:
from automationml import attribute

repaired = document.model_copy(deep=True)
repaired_cell = repaired.instance_hierarchies[0].internal_elements[0]
repaired_robot, repaired_vision = repaired_cell.internal_elements


In [ ]:
repaired_robot.attributes.append(
    attribute(
        "cycleTime", 12, unit="s", data_type="xs:double",
        ref_attribute_type="LineAttributeTypes/CycleTime",
    )
)


In [ ]:
repaired_vision.attributes.append(
    attribute(
        "cycleTime", 6, unit="s", data_type="xs:double",
        ref_attribute_type="LineAttributeTypes/CycleTime",
    )
)

remaining = repaired.caex_validation_issues()
assert Counter(issue.code for issue in remaining) == Counter(
    {"missing-class-interface": 3}
)
print("Attribute warnings repaired; three interface warnings remain.")


## 7. Repair RobotArm's class interfaces

Interface IDs belong to occurrences. Their class paths preserve the
shared interface meaning.


In [ ]:
from automationml import external_interface

repaired_robot.external_interfaces.append(
    external_interface(
        "Power", id="medium-robot-power",
        ref_base_class_path="AutomationMLInterfaceClassLib/PowerPort",
    )
)


In [ ]:
repaired_robot.external_interfaces.append(
    external_interface(
        "MaterialOut", id="medium-robot-material-out",
        ref_base_class_path="AutomationMLInterfaceClassLib/MaterialPort",
    )
)


## 8. Repair VisionGate's remaining interface


In [ ]:
repaired_vision.external_interfaces.append(
    external_interface(
        "MaterialOut", id="medium-quality-material-out",
        ref_base_class_path="AutomationMLInterfaceClassLib/MaterialPort",
    )
)

repaired_issues = repaired.caex_validation_issues(strict_xsd=True)
assert repaired_issues == []
print("The intended class shape is now fully materialized.")


## 9. Add a link between repaired interfaces

The cell owns both children, so it also owns their InternalLink.


In [ ]:
from automationml import InternalLink

repaired_cell.internal_links.append(
    InternalLink(
        name="RobotToVision",
        ref_partner_side_a="medium-cell-robot:MaterialOut",
        ref_partner_side_b="medium-cell-quality:MaterialIn",
    )
)

assert repaired.caex_validation_issues(strict_xsd=True) == []
print("RobotToVision resolves both interface partners.")


## 10. Compare a real error

We break a copy of the link. Unlike an intentionally omitted class
member, a missing partner cannot describe a usable connection.


In [ ]:
broken = repaired.model_copy(deep=True)
broken_link = broken.instance_hierarchies[0].internal_elements[0].internal_links[-1]
broken_link.ref_partner_side_b = "medium-cell-quality:MissingPort"

broken_issues = broken.caex_validation_issues()
for issue in broken_issues:
    print(issue.severity.upper(), issue.code, issue.target)


In [ ]:
assert Counter(issue.code for issue in broken_issues) == Counter(
    {"unresolved-internal-link-partner": 1}
)
assert broken_issues[0].severity == "error"
print("The unresolved partner is correctly classified as an error.")


## Your turn

Remove one different member from a copy of `repaired`. Predict the
issue code before validating. Then decide whether your own project
would accept that warning or require full materialization.

## Takeaways

- Effective class shape combines local and inherited members.
- Missing materialization is a warning because 150% classes can be intentional.
- Structured issue codes make expectations testable and navigable.
- Unresolved class paths and link partners are errors.
- The policy decision belongs to the project; the validator supplies evidence.
